# Inferencia RF en curva unica + gate a ajuste bayesiano

Esta libreta toma una curva salida de fotometria (como en la libreta 4), la adapta al formato del modelo RF, ejecuta clasificacion y, si sale positiva, prepara y dispara el flujo bayesiano.

## 1. Imports y contexto de proyecto

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from classifier.code.inference import load_model_bundle, extract_features_from_csv, predict_single

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## 2. Configuracion de entrada (curva detrended_masked)

In [ ]:
# Modelo RF elegido para produccion
RF_ARTIFACTS = PROJECT_ROOT / "src/classifier/artifacts/rf_final"

# Salida esperada de la libreta 4 (fotometria)
PHOTOMETRY_CURVE_PATH = PROJECT_ROOT / "data/photometry/differential_light_curve_detrended_masked.csv"

# Alternativa si prefieres leer el export dentro de src/photometry
# PHOTOMETRY_CURVE_PATH = PROJECT_ROOT / "src/photometry/differential_light_curve_detrended_masked.csv"

# Metadata de objeto para la fase bayesiana
TARGET_NAME = "target_001"

# Rutas para pipeline de curva unica
SINGLE_DIR = PROJECT_ROOT / "src/classifier/data/single_inference"
SINGLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_INPUT_CURVE = SINGLE_DIR / "curve_for_rf.csv"

# Literatura (opcional): CSV con columnas recomendadas
# target,param,value,sigma,source
LITERATURE_PATH = PROJECT_ROOT / "data/literature/transit_parameters_literature.csv"

print(f"RF artifacts: {RF_ARTIFACTS} | exists={RF_ARTIFACTS.exists()}")
print(f"Curva fotometria: {PHOTOMETRY_CURVE_PATH} | exists={PHOTOMETRY_CURVE_PATH.exists()}")
print(f"Curva para RF: {MODEL_INPUT_CURVE}")
print(f"Literatura: {LITERATURE_PATH} | exists={LITERATURE_PATH.exists()}")

## 3. Adaptar curva de fotometria al formato del modelo RF

In [ ]:
if not PHOTOMETRY_CURVE_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro la curva de fotometria: {PHOTOMETRY_CURVE_PATH}\n"
        "Ejecuta la libreta 4 y ajusta PHOTOMETRY_CURVE_PATH."
    )

lc = pd.read_csv(PHOTOMETRY_CURVE_PATH)

time_col = "time_jd" if "time_jd" in lc.columns and lc["time_jd"].notna().any() else "frame_index"

# Compatibilidad: si existe detrended_masked la usamos; si no, usamos detrended_flux.
if "detrended_masked" in lc.columns:
    flux_col_in = "detrended_masked"
elif "detrended_flux" in lc.columns:
    flux_col_in = "detrended_flux"
else:
    raise ValueError("No se encontro columna de flujo detrended (detrended_masked o detrended_flux).")

rf_curve = lc[[time_col, flux_col_in]].copy()
rf_curve = rf_curve.rename(columns={time_col: "time_jd", flux_col_in: "detrended_flux"})
rf_curve = rf_curve.replace([np.inf, -np.inf], np.nan).dropna(subset=["time_jd", "detrended_flux"])

if len(rf_curve) < 20:
    raise ValueError(f"La curva tiene {len(rf_curve)} puntos validos; se requieren al menos 20.")

rf_curve.to_csv(MODEL_INPUT_CURVE, index=False)
print(f"Curva adaptada guardada en: {MODEL_INPUT_CURVE}")
print(f"Puntos validos: {len(rf_curve)}")
display(rf_curve.head(10))

## 4. Inferencia de una curva con Random Forest

In [ ]:
bundle_rf = load_model_bundle(RF_ARTIFACTS, name="rf")
features = extract_features_from_csv(MODEL_INPUT_CURVE, flux_column="detrended_flux", time_column="time_jd")
pred = predict_single(bundle_rf, features)

prediction_row = {
    "target": TARGET_NAME,
    "curve_path": str(MODEL_INPUT_CURVE),
    "model": "rf",
    "threshold": pred["threshold"],
    "prob_positive": pred["prob_positive"],
    "label_pred": pred["label_pred"],
}

pred_df = pd.DataFrame([prediction_row])
display(pred_df)

is_positive = int(pred["label_pred"]) == 1
print("Resultado RF: POSITIVA -> pasa a ajuste bayesiano" if is_positive else "Resultado RF: NEGATIVA -> no pasa a ajuste bayesiano")

## 5. Gate: preparar y ejecutar (placeholder) ajuste bayesiano

In [ ]:
BAYESIAN_ROOT = PROJECT_ROOT / "src/bayesian"
BAYESIAN_INPUT_DIR = BAYESIAN_ROOT / "data/input"
BAYESIAN_RESULTS_DIR = BAYESIAN_ROOT / "results"
BAYESIAN_INPUT_DIR.mkdir(parents=True, exist_ok=True)
BAYESIAN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BAYESIAN_CURVE_INPUT = BAYESIAN_INPUT_DIR / f"{TARGET_NAME}_curve.csv"
BAYESIAN_META_INPUT = BAYESIAN_INPUT_DIR / f"{TARGET_NAME}_metadata.json"

if is_positive:
    rf_curve.to_csv(BAYESIAN_CURVE_INPUT, index=False)

    meta = {
        "target": TARGET_NAME,
        "rf_prediction": prediction_row,
        "source_curve": str(PHOTOMETRY_CURVE_PATH),
        "rf_input_curve": str(MODEL_INPUT_CURVE),
        "literature_path": str(LITERATURE_PATH),
    }
    BAYESIAN_META_INPUT.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print(f"Input bayesiano preparado: {BAYESIAN_CURVE_INPUT}")
    print(f"Metadata bayesiana: {BAYESIAN_META_INPUT}")

    from bayesian.pipeline import run_bayesian_fit

    try:
        bayes_result = run_bayesian_fit(
            curve_path=BAYESIAN_CURVE_INPUT,
            target_name=TARGET_NAME,
            output_dir=BAYESIAN_RESULTS_DIR,
            config={"sampler": "TODO"},
        )
        print("Ajuste bayesiano ejecutado")
        display(pd.DataFrame([bayes_result]))
    except NotImplementedError as exc:
        print(f"[PENDIENTE] {exc}")
else:
    print("No se ejecuta bayesiano porque la curva no fue clasificada como positiva por RF.")

## 6. Comparacion con literatura (estructura lista)

In [ ]:
if not LITERATURE_PATH.exists():
    print("No existe aun el CSV de literatura.")
    print("Crea data/literature/transit_parameters_literature.csv con columnas:")
    print("target,param,value,sigma,source")
else:
    lit_df = pd.read_csv(LITERATURE_PATH)
    lit_target = lit_df[lit_df["target"].astype(str) == TARGET_NAME].copy()
    print(f"Filas literatura para {TARGET_NAME}: {len(lit_target)}")
    display(lit_target)

    # Placeholder: cuando el ajuste bayesiano devuelva parametros,
    # aqui se cruza contra lit_target y se computa delta/sigma.
    print("Comparacion bayes-vs-literatura pendiente de implementar en src/bayesian/pipeline.py")